In [ ]:
import sys
import importlib
import pickle
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.simple_env as simple_env
importlib.reload(simple_env)

from src.rl.simple_env import SimpleRuleEnv

stream_dir = root / "data" / "stream" / "swat"
window_state_dir = stream_dir / "window_rule_states"

wid = 1436

with open(window_state_dir / f"window_{wid}_rule_state.pkl", "rb") as f:
    window_state = pickle.load(f)

env = SimpleRuleEnv(window_state, max_steps=1)

# 1) keep 一个攻击规则
env.reset()
_, r_attack_keep, _, info1 = env.step(rule_idx=0, action=0)

# 2) disable 一个攻击规则
env.reset()
_, r_attack_disable, _, info2 = env.step(rule_idx=0, action=1)

# 3) disable 一个正常规则（第10条后开始是正常规则，这里取 rule_idx=10）
env.reset()
_, r_normal_disable, _, info3 = env.step(rule_idx=10, action=1)

print("r_attack_keep:", r_attack_keep, info1)
print("r_attack_disable:", r_attack_disable, info2)
print("r_normal_disable:", r_normal_disable, info3)

In [ ]:
import sys
import importlib
import pickle
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.ac_model as ac_model
import src.rl.obs_utils as obs_utils
importlib.reload(ac_model)
importlib.reload(obs_utils)

from src.rl.ac_model import ActorCriticNet
from src.rl.obs_utils import build_state_vector

stream_dir = root / "data" / "stream" / "swat"
window_state_dir = stream_dir / "window_rule_states"
model_dir = root / "outputs" / "models"
model_dir.mkdir(parents=True, exist_ok=True)

wid = 1436
with open(window_state_dir / f"window_{wid}_rule_state.pkl", "rb") as f:
    window_state = pickle.load(f)

active_mask = window_state["active_mask"]
weights = window_state["weights"]
rule_scores = window_state["rule_scores"]
target_labels = window_state["target_labels"]

# 构造监督数据：攻击规则->keep(0), 正常规则->disable(1)
X_list, y_list = [], []
for rule_idx in range(window_state["num_rules"]):
    state_vec = build_state_vector(
        active_mask, weights, rule_scores, rule_idx, target_labels
    )
    X_list.append(state_vec)
    y_list.append(0 if target_labels[rule_idx] == 1 else 1)

X = torch.tensor(np.array(X_list, dtype=np.float32))
y = torch.tensor(np.array(y_list, dtype=np.int64))

model_warm = ActorCriticNet(state_dim=12, action_dim=2, hidden_dim=64)
optimizer = torch.optim.Adam(model_warm.parameters(), lr=1e-3)

for epoch in range(300):
    logits, values = model_warm(X)
    cls_loss = F.cross_entropy(logits, y)

    optimizer.zero_grad()
    cls_loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        pred = logits.argmax(dim=1)
        acc = (pred == y).float().mean().item()
        print(f"epoch={epoch+1}, cls_loss={cls_loss.item():.6f}, acc={acc:.4f}")

with torch.no_grad():
    logits, _ = model_warm(X)
    pred = logits.argmax(dim=1)
    acc = (pred == y).float().mean().item()

save_path = model_dir / "window_1436_actor_warmstart.pth"
torch.save(model_warm.state_dict(), save_path)

print("final acc:", acc)
print("pred:", pred.tolist())
print("true:", y.tolist())
print("saved:", save_path)

In [ ]:
import sys
import importlib
import pickle
from pathlib import Path
import pandas as pd
import torch

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.ac_model as ac_model
import src.rl.simple_env as simple_env
import src.rl.action_utils as action_utils

importlib.reload(ac_model)
importlib.reload(simple_env)
importlib.reload(action_utils)

from src.rl.ac_model import ActorCriticNet
from src.rl.simple_env import SimpleRuleEnv
from src.rl.action_utils import ACTION_NAMES

stream_dir = root / "data" / "stream" / "swat"
window_state_dir = stream_dir / "window_rule_states"
window_pool_dir = stream_dir / "window_rule_pools"
model_dir = root / "outputs" / "models"

wid = 1436

with open(window_state_dir / f"window_{wid}_rule_state.pkl", "rb") as f:
    window_state = pickle.load(f)

rule_pool_df = pd.read_csv(window_pool_dir / f"window_{wid}_mixed_rule_pool.csv")

model = ActorCriticNet(state_dim=12, action_dim=2, hidden_dim=64)
model.load_state_dict(torch.load(model_dir / "window_1436_actor_warmstart.pth", map_location="cpu"))
model.eval()

env = SimpleRuleEnv(window_state, max_steps=window_state["num_rules"])

state = env.reset()
records = []
total_reward = 0.0

for step in range(window_state["num_rules"]):
    rule_idx = step

    state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
    logits, value = model(state_tensor)
    action = torch.argmax(logits, dim=-1).item()

    next_state, reward, done, info = env.step(rule_idx, action)
    total_reward += reward

    records.append({
        "rule_idx": rule_idx,
        "target_label": int(rule_pool_df.iloc[rule_idx]["target_label"]),
        "formula": rule_pool_df.iloc[rule_idx]["formula"],
        "action_name": ACTION_NAMES[action],
        "reward": reward
    })

    state = next_state
    if done:
        break

eval_df = pd.DataFrame(records)

print(eval_df[["rule_idx", "target_label", "action_name", "reward"]])
print("\n动作统计:")
print(eval_df["action_name"].value_counts())
print("\n按 target_label 分组统计:")
print(pd.crosstab(eval_df["target_label"], eval_df["action_name"]))
print("\ngreedy total_reward:", total_reward)

In [ ]:
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
log_dir = root / "outputs" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

attack_keep_rate = (
    ((eval_df["target_label"] == 1) & (eval_df["action_name"] == "keep")).sum()
    / (eval_df["target_label"] == 1).sum()
)

normal_disable_rate = (
    ((eval_df["target_label"] == 0) & (eval_df["action_name"] == "disable")).sum()
    / (eval_df["target_label"] == 0).sum()
)

selection_accuracy = (
    ((eval_df["target_label"] == 1) & (eval_df["action_name"] == "keep")).sum()
    + ((eval_df["target_label"] == 0) & (eval_df["action_name"] == "disable")).sum()
) / len(eval_df)

result_df = pd.DataFrame([{
    "window_id": 1436,
    "attack_keep_rate": attack_keep_rate,
    "normal_disable_rate": normal_disable_rate,
    "selection_accuracy": selection_accuracy,
    "greedy_total_reward": total_reward
}])

save_path = log_dir / "window_1436_metrics.csv"
result_df.to_csv(save_path, index=False)

print("saved:", save_path)
print(result_df)

In [ ]:
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
log_dir = root / "outputs" / "logs"

df_998 = pd.read_csv(log_dir / "window_998_metrics.csv")
df_1436 = pd.read_csv(log_dir / "window_1436_metrics.csv")

summary_df = pd.concat([df_998, df_1436], axis=0, ignore_index=True)

summary_df["window_type"] = "mixed"

save_path = log_dir / "mixed_windows_summary_v1.csv"
summary_df.to_csv(save_path, index=False)

print("saved:", save_path)
print(summary_df)
print("\nmean selection_accuracy:", summary_df["selection_accuracy"].mean())
print("mean greedy_total_reward:", summary_df["greedy_total_reward"].mean())